In [42]:
import sys

print(sys.version)
print(sys.executable)


3.14.4 (main, Jun 18 2026, 14:25:02) [GCC 15.2.0]
/home/nineleaps/mini-rag-chatbot/rag_env/bin/python


## 1. PDF Text Extraction

The first step in the RAG pipeline is to extract text from the
source PDF.

We use PyPDF to read the PDF page by page and extract the textual
content.

In [43]:
from pypdf import PdfReader

pdf_path = "sample_data/data.pdf"

reader = PdfReader(pdf_path)

print("number of pages:", len(reader.pages))

number of pages: 22


In [44]:
#Extract the text
pages_text = []

for page in reader.pages:
    text = page.extract_text()
    pages_text.append(text)

print(pages_text[0][:1000])

A Beginner’s Guide 
to Data & Analytics


In [45]:
#Convert pages into LangChain Documents
from langchain_core.documents import Document

documents = []

for page_number, text in enumerate(pages_text):
    if text:
        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": pdf_path,
                    "page": page_number + 1
                }
            )
        )

print("number of documents:", len(documents))

number of documents: 22


## 2. Text Chunking

The extracted PDF text is divided into smaller chunks so that
each chunk can be independently converted into an embedding and
retrieved during semantic search.

We use overlapping chunks to preserve context across chunk
boundaries.

In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(documents)
print("number of chunks:", len(chunks))

number of chunks: 51


In [47]:
#inspect the chunks
for i in range(3):
    print(f"\n--- chunk {i+1} ---")
    print(chunks[i].page_content)
    print("metadata:", chunks[i].metadata)


--- chunk 1 ---
A Beginner’s Guide 
to Data & Analytics
metadata: {'source': 'sample_data/data.pdf', 'page': 1}

--- chunk 2 ---
Contents
3 Data Science 
vs. Data Analytics: What’s 
the Difference?
7 Data Literacy 101: 
Familiarizing Yourself with 
the Data Landscape
11 Building Your Data & 
Analytical Skill Set
19 Which Data & Analytics 
Course Is Right for You?
Data is ubiquitous. It’s collected at every purchase made, flight taken, ad clicked, and 
social media post liked—which means it’s never been more accessible to organizations.
Yet, access to data isn’t all it takes to set a business on the path to success; it also takes 
employees who understand and know how to leverage data. There’s now an increased 
demand for data-literate business professionals who can handle, analyze, and interpret 
data to drive decision-making.
metadata: {'source': 'sample_data/data.pdf', 'page': 2}

--- chunk 3 ---
data to drive decision-making. 
“In this world of big data, basic data literacy—the abi

In [48]:
#inspect the sizes
for i, chunk in enumerate(chunks[:10]):
    print(
        f"chunk {i+1}: "
        f"{len(chunk.page_content)} characters | "
        f"page: {chunk.metadata['page']}"
    )

chunk 1: 39 characters | page: 1
chunk 2: 709 characters | page: 2
chunk 3: 758 characters | page: 2
chunk 4: 314 characters | page: 3
chunk 5: 774 characters | page: 4
chunk 6: 738 characters | page: 4
chunk 7: 788 characters | page: 5
chunk 8: 771 characters | page: 5
chunk 9: 771 characters | page: 5
chunk 10: 192 characters | page: 5


## 3. Generate Embeddings

Embeddings convert text into numerical vectors that represent
the semantic meaning of the text.

We generate embeddings for every document chunk so that
semantically similar questions and chunks can be identified
during retrieval.


In [49]:
from langchain_huggingface import HuggingFaceEmbeddings

#Create the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|█████████████| 103/103 [00:00<00:00, 1416.44it/s]


In [50]:
#texting embeddings
test_embedding = embedding_model.embed_query(
    "What is data analytics?"
)
print(type(test_embedding))
print(len(test_embedding))
print(test_embedding[:10])

<class 'list'>
384
[-0.021909672766923904, 0.054780229926109314, -0.14216221868991852, 0.0639025941491127, -0.016469096764922142, -0.06386063247919083, 0.07600279152393341, -0.01985573209822178, -0.033540137112140656, 0.029196957126259804]


## 4. Create Vector Database

The embeddings generated from the document chunks are stored
in a FAISS vector database.

FAISS allows us to efficiently search for chunks that are
semantically similar to a user's question.

In [51]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [52]:
print("number of vectors:", vector_store.index.ntotal)

number of vectors: 51


## 5. Semantic Search

The user question is converted into an embedding and compared
with the embeddings stored in the FAISS vector database.

The most semantically similar chunks are retrieved as context
for the LLM.

In [53]:
#Semantic Retrieval

def retrieve_context(question, k=3):
    results = vector_store.similarity_search_with_score(
        question,
        k=k
    )
    
    return results

In [54]:
#inspect the derived chunks
question = "What is data analytics?"

results = retrieve_context(question)

for i, (result, score) in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print("Page:", result.metadata["page"])
    print("Score:", score)
    print(result.page_content)


--- Result 1 ---
Page: 3
Score: 0.62699425
Data Science 
vs. Data Analytics: 
What’s the 
Difference?
If you’re new to the world of data, two terms you’re likely to encounter 
are “data science” and “data analytics. ” While these terms are 
related, they refer to different things. Here’s an overview of 
what each term means and how it applies to business.

--- Result 2 ---
Page: 6
Score: 0.6729407
6
Data Science vs. Data Analytics: What’s the Difference?
Data Analytics in Business
The main goal of business analytics is to extract meaningful insights from 
data that an organization can use to inform its strategy and, ultimately, 
reach its objectives. Business analytics can be used for:
• Budgeting and forecasting: By assessing a company’s historical 
revenue, sales, and costs data alongside its goals for future growth, 
an analyst can identify the budget and investments required to make 
those goals a reality.
• Risk management: By understanding the likelihood of certain 
business ris

In [55]:
def build_context(results):
    context_parts = []
    
    for result, score in results:
        page = result.metadata["page"]
        
        context_parts.append(
            f"[Page {page}]\n{result.page_content}"
        )
    
    return "\n\n".join(context_parts)


question = "What is data analytics?"

results = retrieve_context(question)

context = build_context(results)

print(context)

[Page 3]
Data Science 
vs. Data Analytics: 
What’s the 
Difference?
If you’re new to the world of data, two terms you’re likely to encounter 
are “data science” and “data analytics. ” While these terms are 
related, they refer to different things. Here’s an overview of 
what each term means and how it applies to business.

[Page 6]
6
Data Science vs. Data Analytics: What’s the Difference?
Data Analytics in Business
The main goal of business analytics is to extract meaningful insights from 
data that an organization can use to inform its strategy and, ultimately, 
reach its objectives. Business analytics can be used for:
• Budgeting and forecasting: By assessing a company’s historical 
revenue, sales, and costs data alongside its goals for future growth, 
an analyst can identify the budget and investments required to make 
those goals a reality.
• Risk management: By understanding the likelihood of certain 
business risks occurring—and their associated expenses—an analyst 
can make cost

## 6. Prompt Engineering

The retrieved chunks are provided to the language model as
context.

The model is instructed to answer only from the retrieved
context. If the information is not present in the context,
the model should say that it could not find the answer in
the provided document.

In [56]:
def create_prompt(question, context):
    prompt = f"""
You are a helpful Data Analytics assistant.

Answer the user's question using ONLY the context provided below.

If the answer cannot be found in the context, say:
"I couldn't find the answer in the provided document."

Do not use outside knowledge.

Context:
{context}

Question:
{question}

Answer:
"""
    
    return prompt

In [57]:
question = "What is data analytics?"

results = retrieve_context(question, k=3)

context = build_context(results)

prompt = create_prompt(question, context)

print(prompt)


You are a helpful Data Analytics assistant.

Answer the user's question using ONLY the context provided below.

If the answer cannot be found in the context, say:
"I couldn't find the answer in the provided document."

Do not use outside knowledge.

Context:
[Page 3]
Data Science 
vs. Data Analytics: 
What’s the 
Difference?
If you’re new to the world of data, two terms you’re likely to encounter 
are “data science” and “data analytics. ” While these terms are 
related, they refer to different things. Here’s an overview of 
what each term means and how it applies to business.

[Page 6]
6
Data Science vs. Data Analytics: What’s the Difference?
Data Analytics in Business
The main goal of business analytics is to extract meaningful insights from 
data that an organization can use to inform its strategy and, ultimately, 
reach its objectives. Business analytics can be used for:
• Budgeting and forecasting: By assessing a company’s historical 
revenue, sales, and costs data alongside its g

In [58]:
from dotenv import load_dotenv
import os

load_dotenv()

print("Gemini API key available:", bool(os.getenv("GOOGLE_API_KEY")))

Gemini API key available: True


In [59]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)


In [60]:
def answer_question(question, k=3):
    
    # retrieve relevant chunks
    results = retrieve_context(question, k=k)
    
    # build context
    context = build_context(results)
    
    # create prompt
    prompt = create_prompt(question, context)
    
    # generate answer using Gemini
    response = llm.invoke(prompt)
    
    # collect source pages
    pages = sorted(
        set(
            result.metadata["page"]
            for result, score in results
        )
    )
    
    return response.text, pages

In [61]:
answer, pages = answer_question(
    "What is data analytics?"
)

print("Answer:")
print(answer)

print("\nSource pages:")
print(pages)

Answer:
Based on the provided document, data analytics refers to the process and practice of analyzing data to answer questions, extract insights, and identify trends.

Source pages:
[3, 4, 6]
